## **Create operational and vehicle interaction features for transportation regression.**

In [4]:
import pandas as pd

def engineer_operational_features(df):

    print(f"🔹 Original Dataset shape: {df.shape}")

    # Create a copy to avoid modifying the original
    df_new = df.copy()

    # ====================
    # 1. Vehicle-Distance Fitness
    # ====================
    print("🔨 Engineering Vehicle-Distance Fitness...")

    # Define vehicle suitability based on typical operational ranges
    # These are based on typical real-world transportation logistics
    vehicle_distance_suitability = {
        'motorcycle': 50,      # Optimal for short urban distances
        'car': 200,            # Mid-range, versatile
        'pickup': 300,         # Similar to van
        'van': 300,            # Good for mid-range deliveries
        'truck': 500           # Optimal for long hauls
    }

    # Calculate mismatch score (0-1, where 0 is perfect match, 1 is worst mismatch)
    def calculate_distance_mismatch(row):
        optimal_distance = vehicle_distance_suitability.get(row['vehicle_type'], 200)
        distance = row['distance_km']

        # Normalized mismatch score
        # Formula: min(1, abs(distance - optimal) / max(distance, optimal))
        mismatch = min(1.0, abs(distance - optimal_distance) / max(distance, optimal_distance, 1))

        # Penalize short-distance trucks and long-distance motorcycles more
        if row['vehicle_type'] == 'truck' and row['distance_km'] < 100:
            mismatch *= 1.5
        elif row['vehicle_type'] in ['motorcycle', 'car'] and row['distance_km'] > 300:
            mismatch *= 1.5

        return min(1.0, mismatch)

    df_new['vehicle_distance_mismatch'] = df_new.apply(calculate_distance_mismatch, axis=1)

    # ====================
    # 2. Vehicle-Traffic Stress
    # ====================
    print("🔨 Engineering Vehicle-Traffic Stress...")

    # Traffic level mapping
    traffic_severity = {'low': 1, 'medium': 2, 'high': 3}

    # Vehicle sensitivity to traffic
    # Higher values = more affected by traffic
    vehicle_traffic_sensitivity = {
        'motorcycle': 1.0,    # Can navigate through traffic
        'car': 1.2,           # Moderately affected
        'pickup': 1.5,        # Less maneuverable
        'van': 1.8,           # Large, less maneuverable
        'truck': 2.0          # Most affected by traffic
    }

    # Traffic delay factors (multipliers)
    traffic_delay_factors = {'low': 1.0, 'medium': 1.5, 'high': 2.5}

    def calculate_traffic_stress(row):
        traffic_level = row['traffic_level'].lower()
        vehicle_type = row['vehicle_type']

        base_stress = traffic_severity.get(traffic_level, 1)
        sensitivity = vehicle_traffic_sensitivity.get(vehicle_type, 1.0)

        # Adjust based on peak hour (compounding effect)
        peak_multiplier = 1.5 if row.get('is_peak_hour', False) else 1.0

        # Calculate stress score (1-5 scale)
        stress = min(5.0, base_stress * sensitivity * peak_multiplier)

        return stress

    df_new['vehicle_traffic_stress'] = df_new.apply(calculate_traffic_stress, axis=1)

    # ====================
    # 3. Weather-Vehicle Risk Index
    # ====================
    print("🔨 Engineering Weather-Vehicle Risk Index...")

    # Weather severity mapping
    weather_severity = {
        'clear': 1,
        'clouds': 1.2,
        'mist': 1.5,
        'rain': 2.0,
        'snow': 3.0,
        'fog': 2.5
    }

    # Vehicle weather vulnerability
    # Higher values = more vulnerable to weather
    vehicle_weather_vulnerability = {
        'motorcycle': 2.5,    # Highly vulnerable
        'car': 1.5,           # Moderately vulnerable
        'pickup': 1.8,        # Similar to van
        'van': 1.8,           # Large but stable
        'truck': 1.2          # Least vulnerable (designed for all conditions)
    }

    # Temperature adjustment factor
    def temperature_adjustment(temp):
        if temp < 0:  # Freezing conditions
            return 1.8
        elif temp < 5:  # Very cold
            return 1.5
        elif temp > 30:  # Very hot
            return 1.4
        elif temp > 25:  # Hot
            return 1.2
        else:
            return 1.0

    def calculate_weather_risk(row):
        weather = row.get('weather', 'clear').lower()
        vehicle_type = row['vehicle_type']
        temp = row.get('temperature', 15)

        # Base risk from weather
        base_risk = weather_severity.get(weather, 1.0)

        # Vehicle vulnerability
        vulnerability = vehicle_weather_vulnerability.get(vehicle_type, 1.0)

        # Temperature adjustment
        temp_factor = temperature_adjustment(temp)

        # Nighttime makes weather risks worse
        night_factor = 1.3 if row.get('is_night', False) else 1.0

        # Calculate final risk (1-10 scale)
        risk = min(10.0, base_risk * vulnerability * temp_factor * night_factor)

        return risk

    df_new['vehicle_weather_risk'] = df_new.apply(calculate_weather_risk, axis=1)

    # ====================
    # 4. Vehicle Time-of-Day Efficiency
    # ====================
    print("🔨 Engineering Vehicle Time-of-Day Efficiency...")

    # Vehicle efficiency by time of day
    # Higher values = more efficient during that time
    vehicle_time_efficiency_scores = {
        'motorcycle': {
            'night': 0.7,    # Less efficient at night (visibility)
            'peak': 0.6,     # Good in traffic, but congestion reduces overall efficiency
            'off_peak': 0.9, # Most efficient
            'weekend': 0.8   # Moderate efficiency
        },
        'car': {
            'night': 0.8,
            'peak': 0.5,
            'off_peak': 0.9,
            'weekend': 0.85
        },
        'pickup': {
            'night': 0.85,
            'peak': 0.4,
            'off_peak': 0.9,
            'weekend': 0.8
        },
        'van': {
            'night': 0.9,    # Efficient at night (less traffic)
            'peak': 0.3,     # Very inefficient in peak traffic
            'off_peak': 0.8,
            'weekend': 0.75
        },
        'truck': {
            'night': 1.0,    # Most efficient at night
            'peak': 0.2,     # Very inefficient
            'off_peak': 0.7,
            'weekend': 0.6   # Restricted hours in some areas
        }
    }

    def calculate_time_efficiency(row):
        vehicle_type = row['vehicle_type']
        hour = row.get('order_hour', 12)
        is_peak = row.get('is_peak_hour', False)
        is_night = row.get('is_night', False)
        is_weekend = row.get('is_weekend', False)

        # Determine time category
        if is_night:
            time_cat = 'night'
        elif is_peak:
            time_cat = 'peak'
        elif is_weekend:
            time_cat = 'weekend'
        else:
            time_cat = 'off_peak'

        # Get base efficiency score
        vehicle_scores = vehicle_time_efficiency_scores.get(vehicle_type,
                                                          {'night': 0.8, 'peak': 0.5, 'off_peak': 0.8, 'weekend': 0.8})
        base_efficiency = vehicle_scores.get(time_cat, 0.8)

        # Adjust based on actual hour (fine-tuning)
        hour_adjustment = 1.0
        if 22 <= hour <= 23 or 0 <= hour <= 5:  # Late night
            hour_adjustment = 1.1 if vehicle_type in ['truck', 'van'] else 0.9
        elif 6 <= hour <= 9:  # Morning rush
            hour_adjustment = 0.8 if vehicle_type in ['truck', 'van'] else 0.9
        elif 17 <= hour <= 19:  # Evening rush
            hour_adjustment = 0.7 if vehicle_type in ['truck', 'van'] else 0.85

        # Final efficiency score (0-1 scale)
        efficiency = min(1.0, max(0.1, base_efficiency * hour_adjustment))

        return efficiency

    df_new['vehicle_time_efficiency'] = df_new.apply(calculate_time_efficiency, axis=1)

    # ====================
    # 5. Operational Stress Index
    # ====================
    print("🔨 Engineering Operational Stress Index...")

    def calculate_stress_index(row):
        """
        Composite operational stress indicator combining:
        - Traffic stress
        - Weather risk
        - Time efficiency (inverse)
        - Distance mismatch
        """
        # Normalize components to similar scales (0-1)
        traffic_norm = min(1.0, row['vehicle_traffic_stress'] / 5.0)
        weather_norm = min(1.0, row['vehicle_weather_risk'] / 10.0)
        inefficiency = 1.0 - row['vehicle_time_efficiency']  # Convert efficiency to inefficiency
        distance_mismatch = row['vehicle_distance_mismatch']

        # Weights based on operational impact analysis
        weights = {
            'traffic': 0.35,
            'weather': 0.25,
            'time_inefficiency': 0.25,
            'distance_mismatch': 0.15
        }

        # Calculate weighted stress index
        stress_index = (
            traffic_norm * weights['traffic'] +
            weather_norm * weights['weather'] +
            inefficiency * weights['time_inefficiency'] +
            distance_mismatch * weights['distance_mismatch']
        )

        # Scale to 0-1 range
        return min(1.0, max(0.0, stress_index))

    df_new['operational_stress_index'] = df_new.apply(calculate_stress_index, axis=1)

    # ====================
    # Validation
    # ====================
    print("\n✅ Feature Engineering Complete!")
    print(f"🔹 New Dataset shape: {df_new.shape}")

    # Check for missing values
    new_features = [
        'vehicle_distance_mismatch',
        'vehicle_traffic_stress',
        'vehicle_weather_risk',
        'vehicle_time_efficiency',
        'operational_stress_index'
    ]

    missing_counts = df_new[new_features].isnull().sum()
    print(f"🔹 Missing values in new features:\n{missing_counts}")

    # Check correlation with target (if target exists)
    if 'delivery_time_hours' in df_new.columns:
        correlations = df_new[new_features + ['delivery_time_hours']].corr()['delivery_time_hours'].drop('delivery_time_hours')
        print(f"\n🔹 Correlation of new features with target:")
        for feat, corr in correlations.items():
            print(f"   {feat}: {corr:.4f}")
    else:
        print("⚠️  Target column not found for correlation calculation")

    return df_new



📁 Loading feature_data_v3.csv...
🔹 Original dataset shape: (69926, 35)
🔨 Engineering Vehicle-Distance Fitness...
🔨 Engineering Vehicle-Traffic Stress...
🔨 Engineering Weather-Vehicle Risk Index...
🔨 Engineering Vehicle Time-of-Day Efficiency...
🔨 Engineering Operational Stress Index...

✅ Feature Engineering Complete!
🔹 New dataset shape: (69926, 40)
🔹 Missing values in new features:
vehicle_distance_mismatch    0
vehicle_traffic_stress       0
vehicle_weather_risk         0
vehicle_time_efficiency      0
operational_stress_index     0
dtype: int64

🔹 Correlation of new features with target:
   vehicle_distance_mismatch: 0.3747
   vehicle_traffic_stress: 0.1726
   vehicle_weather_risk: 0.1207
   vehicle_time_efficiency: -0.1424
   operational_stress_index: 0.3252

💾 Dataset saved to: dataset/feature_data_v4.csv

🎯 FEATURE SUMMARY

1️⃣ Vehicle-Distance Fitness (vehicle_distance_mismatch)
   • What: Measures how suitable a vehicle is for the route distance
   • Range: 0-1 (0 = perfect ma

## **Save Dataset**

In [ ]:
def save_dataset(df, filename):
    """Save the engineered Dataset."""
    df.to_csv(filename, index=False)
    print(f"\n💾 Dataset saved to: {filename}")

def print_feature_summary():
    """Print a summary of the new features."""
    print("\n" + "="*60)
    print("🎯 FEATURE SUMMARY")
    print("="*60)
    print("""
1️⃣ Vehicle-Distance Fitness (vehicle_distance_mismatch)
   • What: Measures how suitable a vehicle is for the route distance
   • Range: 0-1 (0 = perfect match, 1 = worst mismatch)
   • Logic: Compares actual distance with vehicle's optimal operational range

2️⃣ Vehicle-Traffic Stress (vehicle_traffic_stress)
   • What: Quantifies how much traffic affects different vehicle types
   • Range: 1-5 (1 = minimal stress, 5 = maximum stress)
   • Logic: Traffic severity × vehicle sensitivity × peak hour multiplier

3️⃣ Weather-Vehicle Risk Index (vehicle_weather_risk)
   • What: Combines weather conditions with vehicle vulnerability
   • Range: 1-10 (1 = minimal risk, 10 = maximum risk)
   • Logic: Weather severity × vehicle vulnerability × temperature factor × night factor

4️⃣ Vehicle Time-of-Day Efficiency (vehicle_time_efficiency)
   • What: Captures how well vehicles perform at different times
   • Range: 0.1-1.0 (0.1 = least efficient, 1.0 = most efficient)
   • Logic: Based on vehicle type and time category (night/peak/off-peak/weekend)

5️⃣ Operational Stress Index (operational_stress_index)
   • What: Composite indicator of overall operational difficulty
   • Range: 0-1 (0 = no stress, 1 = maximum stress)
   • Logic: Weighted combination of traffic, weather, time efficiency, and distance mismatch
    """)
    print("="*60)


## **Main Execution**

In [ ]:
if __name__ == "__main__":
    # Load the Dataset
    print("📁 Loading feature_data_v3.csv...")
    try:
        df = pd.read_csv('feature_data_v3.csv')
    except FileNotFoundError:
        # Try alternative path
        df = pd.read_csv('../Dataset/feature_data_v3.csv')

    # Engineer new features
    df_engineered = engineer_operational_features(df)

    # Save the new Dataset
    save_dataset(df_engineered, '../Dataset/feature_data_v4.csv')

    # Print feature summary
    print_feature_summary()

    # Display sample of new features
    print("\n📊 Sample of new features (first 5 rows):")
    sample_cols = [
        'vehicle_type', 'distance_km', 'traffic_level', 'weather',
        'vehicle_distance_mismatch', 'vehicle_traffic_stress',
        'vehicle_weather_risk', 'vehicle_time_efficiency',
        'operational_stress_index'
    ]

    # Select only existing columns
    existing_cols = [col for col in sample_cols if col in df_engineered.columns]
    print(df_engineered[existing_cols].head())

    print("\n✅ Operational feature engineering completed successfully!")